# IDEA Model Colab Template

This notebook is a user-friendly wrapper around the IDEA command-line pipeline.
It supports:

- Optional raw PDB preprocessing
- Strategy 1: original IDEA gamma training + testing energy
- Strategy 2: ridge-refined gamma from `dna_half.seq` + `exp.txt` + testing energy

Run the cells from top to bottom. Files live only in the Colab session unless you download them.

## 0. Setup

Recommended access is a public GitHub repo URL. If the repo is not on GitHub yet, set `USE_UPLOADED_ZIP = True` and upload a zip of the current `IDEA_Model` folder.

In [ ]:
#@title Setup IDEA_Model in this Colab session
REPO_URL = "https://github.com/YOUR_USERNAME/IDEA_Model.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
USE_UPLOADED_ZIP = False #@param {type:"boolean"}
INSTALL_DEPENDENCIES = True #@param {type:"boolean"}
RESET_EXISTING_REPO = False #@param {type:"boolean"}

import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

WORKDIR = Path("/content") if IN_COLAB else Path.cwd()
PROJECT_ROOT = WORKDIR / "IDEA_Model"

if RESET_EXISTING_REPO and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    if USE_UPLOADED_ZIP:
        if not IN_COLAB:
            raise RuntimeError("Zip upload mode is intended for Google Colab.")
        print("Upload a zip file containing the IDEA_Model folder.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No zip file uploaded.")
        zip_name = next(iter(uploaded))
        zip_path = WORKDIR / zip_name
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(WORKDIR)
        candidates = [p for p in WORKDIR.iterdir() if p.is_dir() and p.name == "IDEA_Model"]
        if not candidates:
            candidates = [p for p in WORKDIR.iterdir() if p.is_dir() and (p / "idea" / "cli.py").exists()]
        if not candidates:
            raise RuntimeError("Could not find IDEA_Model after unzipping.")
        if candidates[0] != PROJECT_ROOT:
            candidates[0].rename(PROJECT_ROOT)
    else:
        if "YOUR_USERNAME" in REPO_URL:
            raise RuntimeError("Set REPO_URL to your IDEA_Model GitHub URL, or use USE_UPLOADED_ZIP=True.")
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

if INSTALL_DEPENDENCIES:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "pyyaml", "scikit-learn", "pandas", "scipy", "matplotlib",
        "biopython", "joblib", "mdtraj", "ipywidgets",
    ], check=True)

# Make sure the local package is importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Setup complete.")

## 1. Shared Helpers

Run this once. Later cells use these helpers to upload files, infer PDB IDs, write YAML configs, find outputs, visualize gamma, and create zip downloads.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import yaml

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

PROJECT_ROOT = Path.cwd()
COLAB_INPUTS = PROJECT_ROOT / "colab_inputs"
COLAB_OUTPUTS = PROJECT_ROOT / "colab_outputs"
COLAB_INPUTS.mkdir(exist_ok=True)
COLAB_OUTPUTS.mkdir(exist_ok=True)

RES_TYPE_LABELS = [
    "A", "R", "N", "D", "C", "Q", "E", "G", "H", "I", "L", "K",
    "M", "F", "P", "S", "T", "W", "Y", "V", "DA", "DG", "DC", "DT",
]


def run_cmd(cmd, cwd=PROJECT_ROOT):
    print("$", " ".join(str(x) for x in cmd))
    process = subprocess.Popen(
        [str(x) for x in cmd], cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")


def upload_one_file(dest_dir, suffixes=None, prompt="Upload one file"):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if not IN_COLAB:
        raise RuntimeError("File upload helper requires Google Colab.")
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one file, got {len(uploaded)}")
    name = next(iter(uploaded))
    if suffixes and not any(name.endswith(suffix) for suffix in suffixes):
        raise ValueError(f"Uploaded file must end with one of {suffixes}: {name}")
    src = PROJECT_ROOT / name
    dest = dest_dir / Path(name).name
    shutil.move(str(src), str(dest))
    print("Saved:", dest)
    return dest


def upload_multiple_files(dest_dir, suffixes=None, prompt="Upload one or more files"):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if not IN_COLAB:
        raise RuntimeError("File upload helper requires Google Colab.")
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No files uploaded.")
    paths = []
    for name in uploaded:
        if suffixes and not any(name.endswith(suffix) for suffix in suffixes):
            raise ValueError(f"Uploaded file must end with one of {suffixes}: {name}")
        src = PROJECT_ROOT / name
        dest = dest_dir / Path(name).name
        shutil.move(str(src), str(dest))
        print("Saved:", dest)
        paths.append(dest)
    return paths


def infer_modified_pdb_id(pdb_path):
    name = Path(pdb_path).name
    suffix = "_modified.pdb"
    if not name.endswith(suffix):
        raise ValueError(f"Processed PDB must be named <id>{suffix}; got {name}")
    return name[:-len(suffix)]


def install_training_pdbs(pdb_paths):
    train_dir = PROJECT_ROOT / "training" / "PDBs"
    train_dir.mkdir(parents=True, exist_ok=True)
    ids = []
    for pdb_path in pdb_paths:
        pdb_id = infer_modified_pdb_id(pdb_path)
        shutil.copy2(pdb_path, train_dir / Path(pdb_path).name)
        ids.append(pdb_id)
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate training PDB ids: {ids}")
    return ids


def install_testing_pdb(pdb_path):
    pdb_id = infer_modified_pdb_id(pdb_path)
    test_dir = PROJECT_ROOT / "testing" / "PDBs"
    test_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pdb_path, test_dir / Path(pdb_path).name)
    return pdb_id


def install_processed_pdb_for_strategy1(pdb_path):
    # Backward-compatible helper for single-PDB examples.
    pdb_id = infer_modified_pdb_id(pdb_path)
    install_training_pdbs([pdb_path])
    install_testing_pdb(pdb_path)
    return pdb_id


def install_processed_pdb_for_strategy2(pdb_path):
    return install_testing_pdb(pdb_path)


def write_yaml(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(data, handle, sort_keys=False)
    print("Wrote config:", path)
    print(path.read_text())
    return path


def latest_experiment(root):
    root = Path(root)
    candidates = [p for p in root.iterdir() if p.is_dir()] if root.exists() else []
    if not candidates:
        raise FileNotFoundError(f"No experiment directories under {root}")
    return max(candidates, key=lambda p: p.stat().st_mtime)


def read_artifact_pointer(job_dir, filename):
    path = Path(job_dir) / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return Path(path.read_text().strip())


def first_job_dir(experiment_dir):
    dirs = [p for p in Path(experiment_dir).iterdir() if p.is_dir()]
    if not dirs:
        raise FileNotFoundError(f"No job folders under {experiment_dir}")
    return sorted(dirs)[0]


def load_gamma(path):
    gamma = np.loadtxt(path, dtype=complex, converters={0: lambda s: complex(s.decode().replace("+-", "-"))})
    gamma = np.asarray(gamma).reshape(-1)
    if np.iscomplexobj(gamma):
        if not np.allclose(gamma.imag, 0, atol=1e-8):
            raise ValueError("Gamma has non-zero imaginary components.")
        gamma = gamma.real
    return gamma


def gamma_to_matrix(gamma):
    gamma = np.asarray(gamma).reshape(-1)
    n = 24
    expected = n * (n + 1) // 2
    if gamma.shape[0] != expected:
        raise ValueError(f"Expected gamma length {expected}, got {gamma.shape[0]}")
    mat = np.zeros((n, n), dtype=float)
    idx = 0
    for i in range(n):
        for j in range(i, n):
            mat[i, j] = gamma[idx]
            mat[j, i] = gamma[idx]
            idx += 1
    return mat


def plot_gamma_heatmap(gamma_path, title="Gamma", scale_mode="adaptive_p95", manual_abs_max=None):
    import matplotlib.pyplot as plt
    gamma = load_gamma(gamma_path)
    mat = gamma_to_matrix(gamma)
    abs_vals = np.abs(mat[np.isfinite(mat)])
    if scale_mode == "adaptive_p95":
        vmax = float(np.percentile(abs_vals, 95)) if abs_vals.size else 1.0
    elif scale_mode == "adaptive_p99":
        vmax = float(np.percentile(abs_vals, 99)) if abs_vals.size else 1.0
    elif scale_mode == "minmax":
        vmax = float(np.max(abs_vals)) if abs_vals.size else 1.0
    elif scale_mode == "manual":
        if manual_abs_max is None:
            raise ValueError("manual_abs_max is required for manual scale.")
        vmax = float(manual_abs_max)
    else:
        raise ValueError(f"Unknown scale_mode: {scale_mode}")
    if vmax <= 0:
        vmax = 1.0
    plt.figure(figsize=(10, 8))
    im = plt.imshow(mat, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(24), RES_TYPE_LABELS, rotation=90)
    plt.yticks(range(24), RES_TYPE_LABELS)
    plt.title(f"{title}\nscale={scale_mode}, abs max={vmax:.4g}")
    plt.tight_layout()
    plt.show()


def find_gamma_candidates(preferred=None):
    candidates = []
    seen = set()
    def add(path, label):
        path = Path(path)
        if path.exists() and path not in seen:
            seen.add(path)
            candidates.append((label, str(path)))
    if preferred:
        add(preferred, f"current: {Path(preferred).parent.name}")
    for path in sorted((PROJECT_ROOT / "runs" / "cache" / "training").glob("*/gamma_filtered.txt"), key=lambda p: p.stat().st_mtime, reverse=True):
        add(path, f"cached: {path.parent.name}")
    for path in sorted((PROJECT_ROOT / "runs_strategy2" / "cache" / "ridge").glob("*/gamma_refined.txt"), key=lambda p: p.stat().st_mtime, reverse=True):
        add(path, f"strategy2: {path.parent.name}")
    return candidates


def zip_and_download(paths, zip_name):
    zip_path = COLAB_OUTPUTS / zip_name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in paths:
            path = Path(path)
            if not path.exists():
                continue
            if path.is_dir():
                for child in path.rglob("*"):
                    if child.is_file():
                        try:
                            arcname = child.relative_to(PROJECT_ROOT)
                        except ValueError:
                            arcname = Path(path.name) / child.relative_to(path)
                        zf.write(child, arcname=str(arcname))
            else:
                try:
                    arcname = path.relative_to(PROJECT_ROOT)
                except ValueError:
                    arcname = path.name
                zf.write(path, arcname=str(arcname))
    print("Created:", zip_path)
    if IN_COLAB:
        files.download(str(zip_path))
    return zip_path

print("Helpers loaded. Project root:", PROJECT_ROOT)

## 2. Optional Raw PDB Preprocessing

Skip this if you already have a processed `<id>_modified.pdb` file.

Upload a raw `.pdb`, `.pdb1`, or `.ent` file. The preprocessing step writes processed files into both `training/PDBs/` and `testing/PDBs/`, then offers a download zip.

In [ ]:
#@title Optional: upload raw PDB and preprocess it
PREPROCESS_DNA_MODE = "auto" #@param ["auto", "ss", "ds"]
DOWNLOAD_PROCESSED_PDB = True #@param {type:"boolean"}

raw_dir = COLAB_INPUTS / "raw_pdbs"
raw_pdb = upload_one_file(raw_dir, suffixes=(".pdb", ".pdb1", ".ent"), prompt="Upload raw PDB for preprocessing")
raw_id = raw_pdb.stem
run_cmd([
    sys.executable, "-m", "idea.cli", "prepare-data",
    "--data-dir", raw_dir,
    "--ids", raw_id,
    "--dna-mode", PREPROCESS_DNA_MODE,
])

processed_paths = [
    PROJECT_ROOT / "training" / "PDBs" / f"{raw_id}_modified.pdb",
    PROJECT_ROOT / "testing" / "PDBs" / f"{raw_id}_modified.pdb",
]
print("Processed files:")
for p in processed_paths:
    print("-", p, "exists=", p.exists())

if DOWNLOAD_PROCESSED_PDB:
    zip_and_download(processed_paths, f"{raw_id}_processed_pdbs.zip")

## 3. Strategy 1: Upload Inputs

Upload one or more training PDBs named `<id>_modified.pdb`, then upload exactly one testing template PDB named `<id>_modified.pdb`, then upload the `dna_half.seq` file for testing.

For Colab free-tier RAM, start with smaller decoy counts in the next parameter cell. The manuscript-scale defaults can be too large for Colab and may cause `optimize_gamma.py` to be killed.

In [ ]:
#@title Strategy 1 upload: training PDB(s), testing PDB, dna_half.seq
s1_training_upload_dir = COLAB_INPUTS / "strategy1" / "training_pdbs"
s1_testing_upload_dir = COLAB_INPUTS / "strategy1" / "testing_pdb"
s1_seq_upload_dir = COLAB_INPUTS / "strategy1" / "sequences"

s1_training_pdb_paths = upload_multiple_files(
    s1_training_upload_dir,
    suffixes=("_modified.pdb",),
    prompt="Upload one or more Strategy1 training PDBs named <id>_modified.pdb",
)
STRATEGY1_TRAINING_IDS = install_training_pdbs(s1_training_pdb_paths)

s1_testing_pdb_path = upload_one_file(
    s1_testing_upload_dir,
    suffixes=("_modified.pdb",),
    prompt="Upload exactly one Strategy1 testing template PDB named <id>_modified.pdb",
)
STRATEGY1_TESTING_ID = install_testing_pdb(s1_testing_pdb_path)

s1_seq_path = upload_one_file(s1_seq_upload_dir, suffixes=(".seq", ".txt"), prompt="Upload Strategy1 dna_half.seq")
seq_dest = PROJECT_ROOT / "testing" / "sequences" / "dna_half.seq"
seq_dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(s1_seq_path, seq_dest)

print("Strategy1 training IDs:", STRATEGY1_TRAINING_IDS)
print("Strategy1 testing ID:", STRATEGY1_TESTING_ID)
print("Installed sequence:", seq_dest)

## 4. Strategy 1: Set Parameters and Generate Config

In [ ]:
#@title Strategy 1 parameters and config generation
S1_RUN_ID = "colab_strategy1" #@param {type:"string"}
S1_DNA_MODE = "ds" #@param ["ss", "ds"]
S1_PROTEIN_CHAIN = "A" #@param {type:"string"}
S1_CONTACT_CUTOFF_NM = 1.2 #@param {type:"number"}
S1_GAMMA_CUTOFF_MODE = 60 #@param {type:"integer"}
S1_PHI_R_MIN = -8.0 #@param {type:"number"}
S1_PHI_R_MAX = 8.0 #@param {type:"number"}
S1_PHI_KAPPA = 0.7 #@param {type:"number"}
S1_PHI_MIN_SEQ_SEP = 10 #@param {type:"integer"}
# Colab-friendly defaults. Increase toward 1000/10000 for production/high-RAM runs.
S1_TRAINING_DECOYS_DNA = 100 #@param {type:"integer"}
S1_TRAINING_DECOYS_PROTEIN = 1000 #@param {type:"integer"}
S1_MAX_TESTING_DECOYS = 1000000 #@param {type:"integer"}

training_label = "_".join(STRATEGY1_TRAINING_IDS)
strategy1_config = {
    "run_id": f"{S1_RUN_ID}_{training_label}_test_{STRATEGY1_TESTING_ID}",
    "defaults": {
        "protein_chain": S1_PROTEIN_CHAIN,
        "contact_cutoff_nm": float(S1_CONTACT_CUTOFF_NM),
        "gamma_cutoff_mode": int(S1_GAMMA_CUTOFF_MODE),
        "phi": {
            "name": "phi_pairwise_contact_well",
            "params": [float(S1_PHI_R_MIN), float(S1_PHI_R_MAX), float(S1_PHI_KAPPA), int(S1_PHI_MIN_SEQ_SEP)],
        },
        "training_decoys": {
            "dna": int(S1_TRAINING_DECOYS_DNA),
            "protein": int(S1_TRAINING_DECOYS_PROTEIN),
        },
        "max_testing_decoys": int(S1_MAX_TESTING_DECOYS),
    },
    "training": {"pdbs": STRATEGY1_TRAINING_IDS},
    "testing": [{"id": STRATEGY1_TESTING_ID, "dna_mode": S1_DNA_MODE}],
}

STRATEGY1_CONFIG_PATH = write_yaml(PROJECT_ROOT / "configs" / "colab_strategy1.yaml", strategy1_config)

In [ ]:
#@title Run Strategy 1 training/testing, then choose gamma for energy
S1_FORCE_RECOMPUTE = False #@param {type:"boolean"}
S1_UPLOAD_CUSTOM_GAMMA = False #@param {type:"boolean"}

from idea.config import load_config
from idea.pipeline import IdeaPipeline

s1_config_loaded = load_config(STRATEGY1_CONFIG_PATH)
s1_pipe = IdeaPipeline(s1_config_loaded)
S1_TRAINING_NAME = next(iter(s1_config_loaded["training_sets"]))
S1_TESTING_NAME = next(iter(s1_config_loaded["testing_sets"]))

print("Training set:", S1_TRAINING_NAME)
print("Testing set:", S1_TESTING_NAME)

S1_TRAINING_ARTIFACT = s1_pipe.train(S1_TRAINING_NAME, force=S1_FORCE_RECOMPUTE)
S1_TESTING_ARTIFACT = s1_pipe.test(S1_TESTING_NAME, force=S1_FORCE_RECOMPUTE)
S1_CURRENT_GAMMA_PATH = S1_TRAINING_ARTIFACT.path / "gamma_filtered.txt"
S1_TESTING_PHI_PATH = S1_TESTING_ARTIFACT.path / "phi_decoys.txt"

if S1_UPLOAD_CUSTOM_GAMMA:
    custom_dir = COLAB_INPUTS / "strategy1" / "custom_gamma"
    S1_CUSTOM_GAMMA_PATH = upload_one_file(custom_dir, suffixes=(".txt",), prompt="Upload custom gamma .txt")
else:
    S1_CUSTOM_GAMMA_PATH = None

import ipywidgets as widgets
from IPython.display import display

candidate_options = []
if S1_CUSTOM_GAMMA_PATH is not None:
    candidate_options.append((f"uploaded custom: {S1_CUSTOM_GAMMA_PATH.name}", str(S1_CUSTOM_GAMMA_PATH)))
candidate_options.extend(find_gamma_candidates(preferred=S1_CURRENT_GAMMA_PATH))
if not candidate_options:
    raise RuntimeError("No gamma candidates found.")

S1_GAMMA_DROPDOWN = widgets.Dropdown(options=candidate_options, description="gamma", layout=widgets.Layout(width="95%"))
display(S1_GAMMA_DROPDOWN)
print("Select the gamma above, then run the next cell to calculate Energy_mg.txt.")
print("Testing phi:", S1_TESTING_PHI_PATH)

In [ ]:
#@title Calculate Strategy 1 energy and visualize selected gamma
from idea.energy import calculate_energy_files

S1_SELECTED_GAMMA_PATH = Path(S1_GAMMA_DROPDOWN.value)
S1_MANUAL_ENERGY_DIR = PROJECT_ROOT / "colab_outputs" / "strategy1_energy" / f"{S1_SELECTED_GAMMA_PATH.parent.name}__{S1_TESTING_ARTIFACT.path.name}"
S1_ENERGY_PATH = calculate_energy_files(S1_SELECTED_GAMMA_PATH, S1_TESTING_PHI_PATH, S1_MANUAL_ENERGY_DIR)
S1_GAMMA_PATH = S1_SELECTED_GAMMA_PATH

print("Selected gamma:", S1_GAMMA_PATH)
print("Energy:", S1_ENERGY_PATH)

S1_SCALE_MODE = "adaptive_p95" #@param ["adaptive_p95", "adaptive_p99", "minmax", "manual"]
S1_MANUAL_ABS_MAX = 1.0 #@param {type:"number"}
plot_gamma_heatmap(S1_GAMMA_PATH, title=f"Strategy1 selected gamma: {S1_GAMMA_PATH.parent.name}", scale_mode=S1_SCALE_MODE, manual_abs_max=S1_MANUAL_ABS_MAX)

In [ ]:
#@title Download Strategy 1 outputs
s1_paths = [
    S1_ENERGY_PATH,
    S1_GAMMA_PATH,
    S1_TRAINING_ARTIFACT.path / "visualize",
    S1_TRAINING_ARTIFACT.manifest_path,
    S1_TESTING_ARTIFACT.manifest_path,
]
zip_and_download(s1_paths, f"strategy1_{STRATEGY1_TESTING_ID}_outputs.zip")

## 5. Strategy 2: Upload Inputs

Upload a processed PDB named `<id>_modified.pdb`, a Strategy2 `dna_half.seq`, and an `exp.txt` file. `dna_half.seq` and `exp.txt` must have the same number of non-empty lines.

In [ ]:
#@title Strategy 2 upload: processed PDB + dna_half.seq + exp.txt
s2_pdb_upload_dir = COLAB_INPUTS / "strategy2" / "pdb"
s2_input_dir = COLAB_INPUTS / "strategy2" / "fit_inputs"

s2_pdb_path = upload_one_file(s2_pdb_upload_dir, suffixes=("_modified.pdb",), prompt="Upload Strategy2 processed PDB named <id>_modified.pdb")
STRATEGY2_PDB_ID = install_processed_pdb_for_strategy2(s2_pdb_path)

s2_seq_uploaded = upload_one_file(s2_input_dir, suffixes=(".seq", ".txt"), prompt="Upload Strategy2 dna_half.seq")
s2_exp_uploaded = upload_one_file(s2_input_dir, suffixes=(".txt",), prompt="Upload Strategy2 exp.txt")

s2_data_dir = PROJECT_ROOT / "data" / "strategy2" / STRATEGY2_PDB_ID
s2_data_dir.mkdir(parents=True, exist_ok=True)
S2_SEQUENCE_PATH = s2_data_dir / "dna_half.seq"
S2_EXP_PATH = s2_data_dir / "exp.txt"
shutil.copy2(s2_seq_uploaded, S2_SEQUENCE_PATH)
shutil.copy2(s2_exp_uploaded, S2_EXP_PATH)

n_seq = len([line for line in S2_SEQUENCE_PATH.read_text().splitlines() if line.strip()])
n_exp = len([line for line in S2_EXP_PATH.read_text().splitlines() if line.strip()])
print("Strategy2 PDB id:", STRATEGY2_PDB_ID)
print("Sequence lines:", n_seq)
print("Exp values:", n_exp)
if n_seq != n_exp:
    raise ValueError("Strategy2 dna_half.seq and exp.txt must have the same number of non-empty lines.")

## 6. Strategy 2: Set Parameters and Generate Config

The alpha UI uses widgets so the alpha input is disabled when `alpha = auto` is checked.

In [ ]:
#@title Strategy 2 parameter widgets
import ipywidgets as widgets
from IPython.display import display

S2_run_id = widgets.Text(value="colab_strategy2", description="run_id")
S2_dna_mode = widgets.Dropdown(options=["ss", "ds"], value="ds", description="dna_mode")
S2_protein_chain = widgets.Text(value="A", description="chain")
S2_contact_cutoff = widgets.FloatText(value=1.2, description="cutoff_nm")
S2_max_testing_decoys = widgets.IntText(value=1000000, description="max_decoys")
S2_alpha_auto = widgets.Checkbox(value=False, description="alpha = auto")
S2_alpha_value = widgets.FloatText(value=1.0, description="alpha")
S2_alpha_min_exp = widgets.FloatText(value=-4.0, description="min_exp")
S2_alpha_max_exp = widgets.FloatText(value=2.0, description="max_exp")
S2_alpha_num = widgets.IntText(value=50, description="alpha_num")
S2_target_sign = widgets.Dropdown(options=["negate", "identity"], value="negate", description="target_sign")
S2_fit_intercept = widgets.Checkbox(value=True, description="fit_intercept")
S2_phi_r_min = widgets.FloatText(value=-8.0, description="r_min")
S2_phi_r_max = widgets.FloatText(value=8.0, description="r_max")
S2_phi_kappa = widgets.FloatText(value=0.7, description="kappa")
S2_phi_min_sep = widgets.IntText(value=10, description="min_sep")

def _toggle_alpha(change=None):
    S2_alpha_value.disabled = S2_alpha_auto.value

S2_alpha_auto.observe(_toggle_alpha, names="value")
_toggle_alpha()

display(widgets.VBox([
    widgets.HTML("<b>Template / IDEA settings</b>"),
    S2_run_id, S2_dna_mode, S2_protein_chain, S2_contact_cutoff, S2_max_testing_decoys,
    widgets.HTML("<b>Phi parameters</b>"),
    widgets.HBox([S2_phi_r_min, S2_phi_r_max, S2_phi_kappa, S2_phi_min_sep]),
    widgets.HTML("<b>Ridge parameters</b>"),
    widgets.HBox([S2_alpha_auto, S2_alpha_value]),
    widgets.HBox([S2_alpha_min_exp, S2_alpha_max_exp, S2_alpha_num]),
    widgets.HBox([S2_target_sign, S2_fit_intercept]),
]))

In [ ]:
#@title Generate Strategy 2 config from widget values
s2_alpha = "auto" if S2_alpha_auto.value else float(S2_alpha_value.value)
strategy2_config = {
    "run_id": f"{S2_run_id.value}_{STRATEGY2_PDB_ID}",
    "defaults": {
        "protein_chain": S2_protein_chain.value,
        "contact_cutoff_nm": float(S2_contact_cutoff.value),
        "phi": {
            "name": "phi_pairwise_contact_well",
            "params": [float(S2_phi_r_min.value), float(S2_phi_r_max.value), float(S2_phi_kappa.value), int(S2_phi_min_sep.value)],
        },
        "max_testing_decoys": int(S2_max_testing_decoys.value),
    },
    "testing": [{"id": STRATEGY2_PDB_ID, "dna_mode": S2_dna_mode.value}],
    "strategy2": {
        "defaults": {
            "alpha": s2_alpha,
            "alpha_grid": {
                "min_exp": float(S2_alpha_min_exp.value),
                "max_exp": float(S2_alpha_max_exp.value),
                "num": int(S2_alpha_num.value),
            },
            "target_sign": S2_target_sign.value,
            "fit_intercept": bool(S2_fit_intercept.value),
        },
        "runs": {
            f"ridge_{STRATEGY2_PDB_ID}": {
                "template_testing": f"test_{STRATEGY2_PDB_ID}",
                "sequences": str(S2_SEQUENCE_PATH.relative_to(PROJECT_ROOT)),
                "exp": str(S2_EXP_PATH.relative_to(PROJECT_ROOT)),
            }
        },
    },
}

STRATEGY2_CONFIG_PATH = write_yaml(PROJECT_ROOT / "configs" / "colab_strategy2.yaml", strategy2_config)
STRATEGY2_RUN_NAME = f"ridge_{STRATEGY2_PDB_ID}"

In [ ]:
#@title Run Strategy 2 pipeline
run_cmd([sys.executable, "-m", "idea.cli", "strategy2", "--config", STRATEGY2_CONFIG_PATH])

S2_EXPERIMENT_DIR = PROJECT_ROOT / "runs_strategy2" / "experiments" / strategy2_config["run_id"]
S2_JOB_DIR = first_job_dir(S2_EXPERIMENT_DIR)
S2_FIT_PHI_ARTIFACT = read_artifact_pointer(S2_JOB_DIR, "fit_phi_artifact.txt")
S2_RIDGE_ARTIFACT = read_artifact_pointer(S2_JOB_DIR, "ridge_artifact.txt")
S2_ENERGY_ARTIFACT = read_artifact_pointer(S2_JOB_DIR, "energy_artifact.txt")
S2_GAMMA_PATH = S2_RIDGE_ARTIFACT / "gamma_refined.txt"
S2_ENERGY_PATH = S2_ENERGY_ARTIFACT / "Energy_mg.txt"

print("Strategy2 experiment:", S2_EXPERIMENT_DIR)
print("Energy:", S2_ENERGY_PATH)
print("Refined gamma:", S2_GAMMA_PATH)

In [ ]:
#@title Strategy 2 refined gamma heatmap
S2_SCALE_MODE = "adaptive_p95" #@param ["adaptive_p95", "adaptive_p99", "minmax", "manual"]
S2_MANUAL_ABS_MAX = 1.0 #@param {type:"number"}
plot_gamma_heatmap(S2_GAMMA_PATH, title=f"Strategy2 refined gamma: {STRATEGY2_PDB_ID}", scale_mode=S2_SCALE_MODE, manual_abs_max=S2_MANUAL_ABS_MAX)

In [ ]:
#@title Download Strategy 2 outputs
s2_paths = [
    S2_ENERGY_PATH,
    S2_GAMMA_PATH,
    S2_EXPERIMENT_DIR / "jobs_summary.csv",
]
zip_and_download(s2_paths, f"strategy2_{STRATEGY2_PDB_ID}_outputs.zip")

## Notes

- Strategy2 final output is only IDEA-style `Energy_mg.txt` (`phi @ gamma_refined`).
- Strategy2 does not output leave-one-out results or model-prediction files.
- Gamma heatmaps use adaptive symmetric scaling by default (`adaptive_p95`) so very large Strategy2 coefficients do not wash out the plot.